# Multi-Omics Analysis Suite - Getting Started

Welcome to the Multi-Omics Analysis Suite! This notebook will guide you through the basic concepts and workflows.

## Table of Contents
1. [Installation & Setup](#setup)
2. [Loading Data](#loading)
3. [Basic Analysis](#analysis)
4. [Visualization](#visualization)
5. [Next Steps](#next-steps)

## 1. Installation & Setup <a name="setup"></a>

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Import MOAS modules
from backend.omics import OmicsRegistry, OmicsCategory
from backend.omics.core import TranscriptomicsModule, ProteomicsModule
from backend.ml.models import RandomForestModel, XGBoostModel
from backend.ml.feature_selection import FeatureSelector

print("Multi-Omics Analysis Suite loaded successfully!")

In [ ]:
# Discover available omics modules
registry = OmicsRegistry()
registry.discover_modules()

print("Available omics modules:")
for category in OmicsCategory:
    modules = registry.list_modules(category)
    if modules:
        print(f"\n{category.value}:")
        for module in modules:
            print(f"  - {module}")

## 2. Loading Data <a name="loading"></a>

The suite supports various data formats including CSV, TSV, GCT, and more.

In [ ]:
# Create sample expression data for demonstration
np.random.seed(42)

n_genes = 1000
n_samples = 50

# Generate expression matrix
expression_data = pd.DataFrame(
    np.random.lognormal(5, 2, (n_genes, n_samples)),
    index=[f"Gene_{i}" for i in range(n_genes)],
    columns=[f"Sample_{i}" for i in range(n_samples)]
)

# Create sample metadata
metadata = pd.DataFrame({
    'sample_id': [f"Sample_{i}" for i in range(n_samples)],
    'group': ['Control'] * 25 + ['Treatment'] * 25,
    'batch': ['Batch1'] * 15 + ['Batch2'] * 35
})

print(f"Expression matrix shape: {expression_data.shape}")
print(f"\nExpression data preview:")
expression_data.head()

In [ ]:
# Initialize transcriptomics module
transcriptomics = TranscriptomicsModule()

print(f"Module: {transcriptomics.name}")
print(f"Category: {transcriptomics.category.value}")
print(f"Description: {transcriptomics.description}")

## 3. Basic Analysis <a name="analysis"></a>

Let's run some basic analyses on our data.

In [ ]:
# Quality Control
from backend.omics.base import DataSource, OmicsData

# Create OmicsData object
omics_data = OmicsData(
    data=expression_data,
    sample_metadata=metadata.set_index('sample_id'),
    feature_metadata=pd.DataFrame(index=expression_data.index),
    omics_type='transcriptomics'
)

# Run QC
qc_report = transcriptomics.quality_control(omics_data)

print("Quality Control Report:")
print(f"  Overall passed: {qc_report.passed}")
for metric in qc_report.metrics:
    print(f"  {metric.name}: {metric.value:.2f} (threshold: {metric.threshold})")

In [ ]:
# Normalization
from backend.omics.base import AnalysisParams

norm_params = AnalysisParams(
    name='normalize',
    parameters={'method': 'quantile'}
)

normalized_data = transcriptomics.normalize(omics_data, norm_params)

print("Data normalized successfully!")
print(f"Normalized data shape: {normalized_data.data.shape}")

In [ ]:
# Run differential expression analysis
de_params = AnalysisParams(
    name='differential_expression',
    parameters={
        'group_column': 'group',
        'control_group': 'Control',
        'treatment_group': 'Treatment',
        'alpha': 0.05,
        'fc_threshold': 1.0
    }
)

de_result = transcriptomics.analyze(normalized_data, de_params)

print(f"Analysis: {de_result.name}")
print(f"Status: {de_result.status}")
print(f"\nTop differentially expressed genes:")
de_result.data.head(10)

## 4. Visualization <a name="visualization"></a>

In [ ]:
# PCA visualization
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Prepare data for PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(expression_data.T)

pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)

# Create PCA dataframe
pca_df = pd.DataFrame(
    pca_coords, 
    columns=['PC1', 'PC2'],
    index=expression_data.columns
)
pca_df['group'] = metadata['group'].values

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
for group in pca_df['group'].unique():
    mask = pca_df['group'] == group
    ax.scatter(
        pca_df.loc[mask, 'PC1'],
        pca_df.loc[mask, 'PC2'],
        label=group,
        s=100,
        alpha=0.7
    )

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA of Expression Data')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Volcano plot
# Simulate DE results for visualization
np.random.seed(42)
log2fc = np.random.normal(0, 1.5, n_genes)
pvalues = np.random.exponential(0.1, n_genes)

# Make some genes significant
sig_idx = np.random.choice(n_genes, 100, replace=False)
log2fc[sig_idx] = np.random.choice([-1, 1], 100) * np.random.uniform(2, 5, 100)
pvalues[sig_idx] = np.random.uniform(1e-10, 0.001, 100)

de_df = pd.DataFrame({
    'gene': [f'Gene_{i}' for i in range(n_genes)],
    'log2FoldChange': log2fc,
    'pvalue': pvalues,
    '-log10(pvalue)': -np.log10(pvalues + 1e-300)
})

# Classify genes
de_df['regulation'] = 'Not significant'
de_df.loc[(de_df['log2FoldChange'] > 1) & (de_df['pvalue'] < 0.05), 'regulation'] = 'Upregulated'
de_df.loc[(de_df['log2FoldChange'] < -1) & (de_df['pvalue'] < 0.05), 'regulation'] = 'Downregulated'

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
colors = {'Not significant': '#999999', 'Upregulated': '#e74c3c', 'Downregulated': '#3498db'}

for reg in ['Not significant', 'Upregulated', 'Downregulated']:
    mask = de_df['regulation'] == reg
    ax.scatter(
        de_df.loc[mask, 'log2FoldChange'],
        de_df.loc[mask, '-log10(pvalue)'],
        c=colors[reg],
        label=f"{reg} ({mask.sum()})",
        alpha=0.6,
        s=20
    )

ax.axhline(-np.log10(0.05), color='gray', linestyle='--', linewidth=1)
ax.axvline(1, color='gray', linestyle='--', linewidth=1)
ax.axvline(-1, color='gray', linestyle='--', linewidth=1)

ax.set_xlabel('log2(Fold Change)')
ax.set_ylabel('-log10(p-value)')
ax.set_title('Volcano Plot')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Next Steps <a name="next-steps"></a>

Now that you've learned the basics, explore more advanced features:

- **02_transcriptomics_analysis.ipynb** - Deep dive into RNA-seq analysis
- **03_multi_omics_integration.ipynb** - Integrate multiple omics datasets
- **04_machine_learning.ipynb** - Train ML models on omics data
- **05_pathway_analysis.ipynb** - Functional enrichment analysis

For more information, see the [documentation](https://github.com/your-repo/multi-omics-suite).